In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Introduction**
Extract, Transform and Load (ETL) operations are of extreme importance in the role of a Data engineer. A data engineer extracts data from multiple sources and different file formats, transforms the extracted data to predefined settings and then loads the data to a database for further processing. In this exercise, you will get hands-on practice of performing these operations.

# **Objectives**

This ETL (Extract, Transform, Load) pipeline is designed to teach students the fundamental concepts of data extraction, transformation, and loading using Python. The pipeline processes data from multiple formats (CSV, JSON, and Excel), transforms the data, and loads it into a structured CSV file. Logging is implemented throughout the pipeline to ensure tracking and debugging.

In [ ]:
import pandas as pd
import json
import logging
from datetime import datetime
import os

def log_progress(message):
    timestamp_format = '%Y-%b-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second
    now = datetime.now() # get current timestamp
    timestamp = now.strftime(timestamp_format)
    with open(log_file,"a") as f:
        f.write(timestamp + ',' + message + '\n')

# Define absolute paths
log_file = os.path.abspath("etl_log.txt")
output_file = os.path.abspath("final_output.csv")

# Explicitly create the file before logging starts
with open(log_file, "w") as f:
    f.write("ETL Log File Initialized\n")

# Setup logging with absolute patgs
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)



def extract_csv(file_path):
    """Extract data from a CSV file."""
    try:
        data = pd.read_csv(file_path)
        logging.info(f"Successfully extracted data from {file_path}")
        return data
    except Exception as e:
        logging.error(f"Error extracting CSV file: {e}")
        return None

def extract_json(file_path):
    """Extract data from a JSON file."""
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
        df = pd.DataFrame(data)
        logging.info(f"Successfully extracted data from {file_path}")
        return df
    except Exception as e:
        logging.error(f"Error extracting JSON file: {e}")
        return None

def extract_excel(file_path, sheet_name="Sheet1"):
    """Extract data from an Excel file."""
    try:
        data = pd.read_excel(file_path, sheet_name=sheet_name)
        logging.info(f"Successfully extracted data from {file_path}")
        return data
    except Exception as e:
        logging.error(f"Error extracting Excel file: {e}")
        return None

def transform_data(sales_df, products_df, customers_df):
    """Transform data by handling missing values, merging, and formatting."""
    try:
        # Fill missing values
        sales_df.fillna({'price': sales_df['price'].mean()}, inplace=True)

        # Convert date format
        sales_df['sale_date'] = pd.to_datetime(sales_df['sale_date']).dt.strftime('%Y-%m-%d')

        # Merge sales with product details
        merged_df = sales_df.merge(products_df, on='product_id', how='left')

        # Merge with customer data
        final_df = merged_df.merge(customers_df, on='customer_id', how='left')

        # Rename columns for consistency
        final_df.columns = final_df.columns.str.lower().str.replace(' ', '_')

        logging.info("Successfully transformed data")
        return final_df
    except Exception as e:
        logging.error(f"Error transforming data: {e}")
        return None

# Ensure the transformed data is saved correctly
def load_data(final_df, output_file):
    """Load transformed data into a CSV file."""
    try:
        final_df.to_csv(output_file, index=False)
        logging.info(f"Successfully loaded data into {output_file}")
        print(f"Data successfully saved at: {output_file}")
    except Exception as e:
        logging.error(f"Error loading data: {e}")

def etl_pipeline():
    log_progress("ETL Job Started")

    log_progress("Extract phase Started")
    sales_df = extract_csv("sales_data.csv")
    products_df = extract_json("product_data.json")
    customers_df = extract_excel("customer_data.xlsx")
    log_progress("Extract phase Ended")

    # Check if all extractions were successful
    if sales_df is not None and products_df is not None and customers_df is not None:
        log_progress("Transform phase Started")
        # Pass individual DataFrames to transform_data
        transformed_data = transform_data(sales_df, products_df, customers_df)
        log_progress("Transform phase Ended")

        if transformed_data is not None:
            log_progress("Load phase Started")
            load_data(transformed_data, output_file) # Pass output_file as second argument
            log_progress("Load phase Ended")

    log_progress("ETL Job Ended")

if __name__ == "__main__":
    etl_pipeline()
    print("ETL process completed. Check etl_log.txt for logs.")



Data successfully saved at: /content/final_output.csv
ETL process completed. Check etl_log.txt for logs.
